# 📊 Module 01: Retinal OCT Dataset Ingestion, Preprocessing & EDA
### Kermany2017 Optical Coherence Tomography (OCT) Dataset Pipeline

This notebook implements:
1. Downloading and loading the Kermany2017 OCT dataset via Hugging Face.
2. Clinical definitions of the 4 retinal diagnostic categories: **CNV, DME, DRUSEN, NORMAL**.
3. Stratified splitting into **Train (70%)**, **Validation (15%)**, and **Test (15%)**.
4. PyTorch `Dataset` and `DataLoader` pipeline with medical image data augmentations.
5. Exploratory Data Analysis (EDA): class frequency distributions and visual gallery of OCT B-scans.

In [ ]:
import os
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from sklearn.model_selection import StratifiedShuffleSplit

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style="whitegrid")

### 1.1 Clinical Categories & Pathology Descriptions

In [ ]:
CLASS_NAMES = ["CNV", "DME", "DRUSEN", "NORMAL"]
NUM_CLASSES = len(CLASS_NAMES)

CLASS_DESCRIPTIONS = {
    "CNV": "Choroidal Neovascularization: Pathologic growth of abnormal blood vessels beneath the retina, causing severe vision loss and subretinal fluid accumulation.",
    "DME": "Diabetic Macular Edema: Accumulation of fluid in the macula due to leaking blood vessels from diabetic retinopathy, leading to retinal thickening.",
    "DRUSEN": "Drusen: Yellow extracellular lipid deposits beneath the retinal pigment epithelium (RPE), characteristic early sign of Age-related Macular Degeneration (AMD).",
    "NORMAL": "Normal Retina: Preserved retinal layer architecture with continuous foveal contour and no intraretinal or subretinal fluid/deposits."
}

for name, desc in CLASS_DESCRIPTIONS.items():
    print(f"[{name}]: {desc}\n")

### 1.2 Data Transforms & Augmentation Pipeline

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB") if hasattr(img, "convert") else img),
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=10),
        transforms.RandomAffine(degrees=0, scale=(0.95, 1.05)),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    eval_transform = transforms.Compose([
        transforms.Lambda(lambda img: img.convert("RGB") if hasattr(img, "convert") else img),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    ])

    return train_transform, eval_transform

print("Transforms defined successfully.")

### 1.3 PyTorch Dataset Class

In [ ]:
class OCTDataset(Dataset):
    """PyTorch Dataset for Retinal OCT Images supporting in-memory bytes, PIL Images, or paths."""
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        item = self.images[idx]
        if isinstance(item, bytes):
            img = Image.open(io.BytesIO(item)).convert("RGB")
        elif isinstance(item, Image.Image):
            img = item.convert("RGB")
        elif isinstance(item, str):
            img = Image.open(item).convert("RGB")
        else:
            raise TypeError(f"Unsupported image type: {type(item)}")

        if self.transform is not None:
            img = self.transform(img)

        label = int(self.labels[idx])
        return img, label

### 1.4 Dataset Loading & Stratified Splitting

In [ ]:
def load_kermany_oct_data(cache_dir=None, val_ratio=0.15, test_ratio=0.15, random_state=42):
    parquet_path = hf_hub_download(
        repo_id="zacharielegault/Kermany2017-OCT",
        filename="data/test-00000-of-00001.parquet",
        repo_type="dataset",
        cache_dir=cache_dir
    )

    table = pq.read_table(parquet_path)
    images_bytes = [table["image"][i]["bytes"].as_py() for i in range(len(table))]
    labels = [table["label"][i].as_py() for i in range(len(table))]

    labels_np = np.array(labels)
    total_samples = len(labels)

    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_state)
    train_val_idx, test_idx = next(sss_test.split(range(total_samples), labels_np))

    val_rel_ratio = val_ratio / (1.0 - test_ratio)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_rel_ratio, random_state=random_state)
    train_sub_idx, val_sub_idx = next(sss_val.split(train_val_idx, labels_np[train_val_idx]))

    train_idx = train_val_idx[train_sub_idx]
    val_idx = train_val_idx[val_sub_idx]

    data_splits = {
        "train_images": [images_bytes[i] for i in train_idx],
        "train_labels": [labels[i] for i in train_idx],
        "val_images": [images_bytes[i] for i in val_idx],
        "val_labels": [labels[i] for i in val_idx],
        "test_images": [images_bytes[i] for i in test_idx],
        "test_labels": [labels[i] for i in test_idx],
    }

    print(f"Dataset successfully loaded: Train={len(train_idx)}, Val={len(val_idx)}, Test={len(test_idx)}")
    return data_splits

splits = load_kermany_oct_data()

### 1.5 Class Distribution Visualization across Splits

In [ ]:
records = []
for split_name, labels in [("Train", splits["train_labels"]), ("Val", splits["val_labels"]), ("Test", splits["test_labels"])]:
    for lbl in labels:
        records.append({"Split": split_name, "Class": CLASS_NAMES[lbl]})
df_split = pd.DataFrame(records)

plt.figure(figsize=(9, 5))
sns.countplot(data=df_split, x="Class", hue="Split", palette="mako")
plt.title("Retinal OCT Dataset Split Class Distribution", fontsize=13, fontweight="bold")
plt.xlabel("OCT Retinal Condition", fontsize=11)
plt.ylabel("Number of Scans", fontsize=11)
plt.grid(axis="y", linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

### 1.6 Visualizing Real Sample OCT Scans

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
samples_by_class = {}
for img_b, lbl in zip(splits["test_images"], splits["test_labels"]):
    if lbl not in samples_by_class:
        samples_by_class[lbl] = img_b
    if len(samples_by_class) == 4:
        break

for i, (lbl, img_b) in enumerate(sorted(samples_by_class.items())):
    img = Image.open(io.BytesIO(img_b)).convert("RGB")
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"{CLASS_NAMES[lbl]}", fontsize=12, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Sample Kermany2017 Retinal OCT Scans Across 4 Classes", fontsize=14, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

### 1.7 DataLoaders Instantiation

In [ ]:
train_tf, eval_tf = get_transforms(img_size=224)
train_ds = OCTDataset(splits["train_images"], splits["train_labels"], transform=train_tf)
val_ds = OCTDataset(splits["val_images"], splits["val_labels"], transform=eval_tf)
test_ds = OCTDataset(splits["test_images"], splits["test_labels"], transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

print(f"DataLoaders ready: {len(train_loader)} train batches, {len(val_loader)} val batches, {len(test_loader)} test batches.")